In [12]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "amici2018social")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Amici_2018_SciRep_SOES.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [13]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)
df['study_id']="amici2018social"
df['experiment_name']="social_inhibition_task_socinh2"

In [14]:

df['Date']= pd.to_datetime(df['Date'],format='%d.%m.%Y')

df['year']= df['Date'].dt.year
df['month']= df['Date'].dt.month
df['day']= df['Date'].dt.day
# df.columns

In [15]:
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['subject'].replace(x, y, inplace=True)
    df['partner'].replace(x, y, inplace=True)
df['dyad']=df.subject.str.cat(df.partner, sep='_')


In [16]:
df=df.rename(columns={"subject": "ape",
    "partner": "ape_2"})

role=[]
role_2=[]
for index, row in df.iterrows():
    if not pd.isna(row['ape']):
        role.append("focal_participant")
    else:
        role.append("")
df = df.assign(role=role)
for index, row in df.iterrows(): 
    if not pd.isna(row['ape_2']):
        role_2.append("partner")
    else:
        role_2.append("")
df = df.assign(role_2=role_2)

In [17]:
df['ape'].replace('', np.nan, inplace=True)
df['condition'].replace(' ', "_", inplace=True, regex=True)
df.dropna(subset=['ape'], inplace=True)

df['ape'] = df['ape'].str.rstrip()
df['ape_2'] = df['ape_2'].str.rstrip()



comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
df.columns

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')
df=df.rename(columns={"species_y": "species"})


In [18]:
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

In [19]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365

    df.rename(columns={"partner position":"partner_position", }, inplace=True)

In [20]:
amici2018social_standardized=df[['study_id', 'experiment_name', 'year', 'month', 'day', 
                                  'participant','age_in_years','sex','role', 
                                  'participant_2','age_in_years_2', 'sex_2', 'role_2', 'species',
       'dyad', 
       'partner_position', 'session', 'trial', 'condition',
       'position_larger_food', 'choice_made', 'largerchoice_yes1_no0']]

In [21]:
comp_out_path_stand = os.path.join(out_pathway, 'amici2018social_standardized.csv')
amici2018social_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [22]:
names =amici2018social_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
amici2018social_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'amici2018social_glossary.csv')
amici2018social_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
